# Clusterização de vinhos 

- **Dataset:** Wine Quality (Vinho Verde)
- **Contexto do problema:** o objetivo édescobrir grupos naturais de vinhos que sejam quimicamente semelhantes entre si, sem informar ao algoritmo o tipo (color) ou a nota de qualidade (quality) de cada vinho.
- **Objetivo da clusterização:** verificar se existem agrupamentos naturais nos dados e, em caso positivo, entender o que caracteriza cada grupo e se esses grupos têm correspondência com informações que já temos (tipo do vinho e nota de qualidade) ou revelam uma estrutura diferente.
- **Algoritmo principal:** K-Means, com o número de clusters definido a partir do Método do Cotovelo e do Silhouette Score. 
- **Variáveis reservadas para interpretação (não usadas na clusterização):** `color` (tipo do vinho) e `quality` (nota de qualidade) 


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors

sns.set_style('whitegrid')

url = "https://raw.githubusercontent.com/klaytoncastro/idp-machinelearning/main/winequality/winequality-merged.csv"
df = pd.read_csv(url)

df.head()

## Exploração inicial do dataset

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Valores nulos por coluna:", df.isnull().sum().sum())
print("Linhas duplicadas:", df.duplicated().sum())
print("\nDistribuição por tipo de vinho (color):")
print(df['color'].value_counts())

Há valores corrompidos em `alcohol` e `density` (magnitudes fisicamente impossíveis, provavelmente erro de parsing/unidade) e 1.176 linhas duplicadas. Como a clusterização é uma técnica baseada em distância, estes problemas são críticos, podendo distorcer a escala de toda a variável após a padronização. Dito isso, foi feita a limpeza dos dados.

In [ ]:
print("Linhas com alcohol > 20:", (df['alcohol'] > 20).sum())
print("Linhas com density > 1.5:", (df['density'] > 1.5).sum())

df = df[(df['alcohol'] <= 20) & (df['density'] <= 1.5)]
df = df.drop_duplicates().reset_index(drop=True)

df['color'] = df['color'].map({'red': 0, 'white': 1})

print("\nFormato final do dataset:", df.shape)
df.describe()

## Seleção das variáveis para a clusterização

**11 variáveis físico-químicas contínuas** (acidez, açúcares, dióxido de enxofre, densidade, pH, sulfatos e álcool). Duas colunas ficam **fora** do conjunto de clusterização:

- `quality`
- `color`


## Matriz de correlação 

In [ ]:
features = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
            'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
            'pH', 'sulphates', 'alcohol']

X = df[features].copy()

plt.figure(figsize=(9, 7))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlação entre as variáveis físico-químicas usadas na clusterização')
plt.tight_layout()
plt.show()

Há correlações fortes entre `free sulfur dioxide` e `total sulfur dioxide` (0,72), entre `residual sugar` e `density` (0,49) e entre `alcohol` e `density` (−0,70). 

## Preparação dos dados: padronização

K-Means usa **distância euclidiana** para agrupar os pontos. As variáveis estão em escalas muito diferentes: `total sulfur dioxide` chega a centenas, enquanto `chlorides` fica na casa dos centésimos . Foi aplicado `StandardScaler` para colocar todas as variáveis na mesma escala (média 0, desvio-padrão 1).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=features)
X_scaled.describe().loc[['mean', 'std', 'min', 'max']]

## Definição do número de clusters (k)

Critérios utilizados para escolher o melhor $k$:

- **Método do Cotovelo (Elbow)** 
- **Silhouette Score** 
- **Davies-Bouldin Index**

In [ ]:
k_range = range(2, 11)
inertias = []
silhouettes = []
dbis = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_k))
    dbis.append(davies_bouldin_score(X_scaled, labels_k))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(list(k_range), inertias, marker='o')
axes[0].set_title('Método do Cotovelo')
axes[0].set_xlabel('k (número de clusters)')
axes[0].set_ylabel('Inércia')

axes[1].plot(list(k_range), silhouettes, marker='o', color='seagreen')
axes[1].set_title('Silhouette Score por k')
axes[1].set_xlabel('k (número de clusters)')
axes[1].set_ylabel('Silhouette Score')

axes[2].plot(list(k_range), dbis, marker='o', color='indianred')
axes[2].set_title('Davies-Bouldin Index por k')
axes[2].set_xlabel('k (número de clusters)')
axes[2].set_ylabel('Davies-Bouldin (menor = melhor)')

plt.tight_layout()
plt.show()

pd.DataFrame({'k': list(k_range), 'inertia': inertias, 'silhouette': silhouettes, 'davies_bouldin': dbis})

## Treinamento do modelo final de clusterização

In [ ]:
K_FINAL = 2  # ajustado conforme a análise de silhouette/cotovelo acima

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print("Tamanho de cada cluster:")
print(df['cluster'].value_counts().sort_index())

## Avaliação dos agrupamentos

In [ ]:
sil = silhouette_score(X_scaled, df['cluster'])
dbi = davies_bouldin_score(X_scaled, df['cluster'])

print(f"Número de clusters: {K_FINAL}")
print(f"Silhouette Score:      {sil:.3f}")
print(f"Davies-Bouldin Index:  {dbi:.3f}")
print(f"Inércia (WCSS):        {kmeans.inertia_:.1f}")

## Visualização dos clusters (redução de dimensionalidade com PCA)

Como o espaço de atributos tem 11 dimensões, utilizei PCA para reduzir a 2 componentes principais e visualizar a separação encontrada pelo K-Means. 

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Variância explicada por PC1 e PC2: {pca.explained_variance_ratio_[0]:.1%} e {pca.explained_variance_ratio_[1]:.1%} "
      f"(total {pca.explained_variance_ratio_.sum():.1%})")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

sc0 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'], cmap='viridis', s=15, alpha=0.6, edgecolor='none')
axes[0].set_title('Colorido pelo cluster (K-Means)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(sc0, ax=axes[0])

sc1 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=df['color'], cmap='coolwarm', s=15, alpha=0.6, edgecolor='none')
axes[1].set_title('Colorido pelo tipo real (0=tinto, 1=branco)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[1])

sc2 = axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=df['quality'], cmap='plasma', s=15, alpha=0.6, edgecolor='none')
axes[2].set_title('Colorido pela nota de qualidade')
axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')
plt.colorbar(sc2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
loadings = pd.Series(pca.components_[0], index=features).sort_values()

plt.figure(figsize=(7, 5))
loadings.plot(kind='barh', color='steelblue')
plt.title('Contribuição de cada variável para o PC1')
plt.xlabel('Carga (loading) no primeiro componente principal')
plt.tight_layout()
plt.show()

loadings

## Caracterização dos clusters

In [ ]:
cluster_means = df.groupby('cluster')[features].mean().T
cluster_means.columns = [f'Cluster {c}' for c in cluster_means.columns]
cluster_means

In [ ]:
centers_z = pd.DataFrame(kmeans.cluster_centers_, columns=features).T
centers_z.columns = [f'Cluster {c}' for c in centers_z.columns]

plt.figure(figsize=(6, 7))
sns.heatmap(centers_z, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Centros dos clusters (variáveis padronizadas)\nvalores > 0: acima da média geral | < 0: abaixo da média geral')
plt.tight_layout()
plt.show()

In [ ]:
top_features = centers_z.assign(diff=lambda d: (d.iloc[:, 0] - d.iloc[:, 1]).abs()).sort_values('diff', ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.ravel(), top_features):
    sns.boxplot(data=df, x='cluster', y=feat, ax=ax, hue='cluster', palette='viridis', legend=False)
    ax.set_title(feat)
    ax.set_xlabel('Cluster')

plt.tight_layout()
plt.show()

print("Variáveis mais discriminativas entre os clusters:", top_features)


Comparacao dos clusters (formados **sem** usar `color` ou `quality`)

In [ ]:
ct_color = pd.crosstab(df['cluster'], df['color'].map({0: 'tinto', 1: 'branco'}), normalize='index') * 100
print("Composição de cada cluster por tipo de vinho (%):")
print(ct_color.round(1))

plt.figure(figsize=(6, 4))
ct_color.plot(kind='bar', stacked=True, ax=plt.gca(), colormap='coolwarm')
plt.title('Composição de cada cluster por tipo de vinho')
plt.ylabel('% dentro do cluster')
plt.xlabel('Cluster')
plt.legend(title='Tipo de vinho')
plt.tight_layout()
plt.show()

In [ ]:
quality_by_cluster = df.groupby('cluster')['quality'].agg(['mean', 'std', 'count'])
print("Nota de qualidade média por cluster:")
print(quality_by_cluster.round(3))

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='cluster', y='quality', hue='cluster', palette='viridis', legend=False)
plt.title('Distribuição da nota de qualidade por cluster')
plt.xlabel('Cluster')
plt.ylabel('Nota de qualidade (quality)')
plt.tight_layout()
plt.show()

## Comparação com um algoritmo baseado em densidade (DBSCAN)

O K-Means assume clusters aproximadamente esféricos e de tamanho semelhante, e exige que $k$ seja escolhido de antemão. O DBSCAN segue uma lógica diferente: agrupa por densidade e decide sozinho o número de clusters, além de conseguir marcar pontos isolados como ruído (label `-1`). Vale testar essa alternativa no mesmo conjunto padronizado — como sugere o [Machine Learning Map](https://scikit-learn.org/stable/machine_learning_map.html), a escolha do algoritmo depende de como os dados estão distribuídos no espaço.

Para escolher o `eps`, usamos o gráfico de distância ao k-ésimo vizinho mais próximo (aqui, `min_samples = 2 * n_features = 22`) e procuramos o "cotovelo" da curva.

In [ ]:
min_samples = 2 * len(features)

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_scaled)
distances, _ = neighbors_fit.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(7, 4))
plt.plot(k_distances)
plt.title(f'Gráfico de distância ao {min_samples}º vizinho mais próximo')
plt.xlabel('Pontos ordenados pela distância')
plt.ylabel(f'Distância ao {min_samples}º vizinho')
plt.tight_layout()
plt.show()

In [ ]:
EPS = 2.6  

dbscan = DBSCAN(eps=EPS, min_samples=min_samples)
labels_db = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = int((labels_db == -1).sum())

print(f"Clusters encontrados pelo DBSCAN: {n_clusters_db}")
print(f"Pontos classificados como ruído: {n_noise_db} ({n_noise_db / len(labels_db):.1%})")

if n_clusters_db >= 2:
    mask = labels_db != -1
    print(f"Silhouette Score (ignorando ruído):     {silhouette_score(X_scaled[mask], labels_db[mask]):.3f}")
    print(f"Davies-Bouldin Index (ignorando ruído): {davies_bouldin_score(X_scaled[mask], labels_db[mask]):.3f}")

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_db, cmap='viridis', s=15, alpha=0.6, edgecolor='none')
plt.title(f'DBSCAN (eps={EPS}, min_samples={min_samples}) — projeção em PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(sc)
plt.show()

## Interpretação dos resultados

1. Qualidade da partição
O K-Means com k=2 chegou a um Silhouette de 0,268 e Davies-Bouldin de 1,617. Isso indica uma separação real entre os grupos, mas moderada, há uma sobreposição, que também aparece no PCA .

2. O que diferencia os dois clusters
O Cluster 0, com 3.842 vinhos, se destaca por níveis de SO₂ livre e total bem mais altos (em torno de 35 e 137), mais açúcar residual (5,8) e menos acidez volátil, cloretos e sulfatos. Já o Cluster 1, com 1.345 vinhos, mostra o padrão inverso: pouco SO₂, pouco açúcar, e mais acidez volátil, cloretos, sulfatos e acidez fixa.

4. Qualidade
Aqui a coisa muda de figura: a qualidade quase não é capturada pelos clusters. A média fica em 5,878 no cluster dos brancos contra 5,575 no dos tintos, uma diferença de  ~0,30, bem menor que o desvio-padrão dentro de cada cluster (≈0,86). Isso bate com o R² fraco (0,287) q

